# NB03 · Semana 7 — DuckDB y Delta Lake sobre el lake

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07**

## Objetivo

Consultar el bronze (Delta Lake en MinIO) con **DuckDB**, explorarlo para encontrar los problemas que tiene que limpiar silver, y ver el **historial de versiones** de Delta (*time travel*).

Cada hallazgo de la sección 2 termina en una decisión de `stg_urlhaus` (ver `docs/entregas/E2_dbt.md`).

In [ ]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BUCKET, BRONZE, STORAGE_OPTIONS, info
info()

In [ ]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

## 1. Leer el bronze

`deltalake` lee la versión vigente de la tabla (solo los archivos que la componen) y DuckDB la consulta con SQL, sin cargarla en ninguna base.

In [ ]:
import duckdb
import pandas as pd
from deltalake import DeltaTable

dt = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS)
bronze = dt.to_pyarrow_table()
con = duckdb.connect()
con.sql("set TimeZone = 'UTC'")          # las fechas de URLhaus están en UTC
con.register("bronze", bronze)
pd.set_option("display.max_colwidth", 80)

print(f"{BRONZE}: versión {dt.version()}, {bronze.num_rows} filas, {bronze.num_columns} columnas")
print(bronze.schema)

## 2. Exploración: qué problemas tiene el dato crudo

### 2.1 ¿Hay URLs repetidas o sin datos clave?

In [ ]:
con.sql("""
    select count(*)                                   as filas,
           count(distinct id)                         as ids_distintos,
           count(*) - count(distinct id)              as filas_repetidas,
           count(*) filter (where id is null)         as sin_id,
           count(*) filter (where date_added is null) as sin_fecha,
           count(*) filter (where tags is null or trim(tags) = '') as sin_tags,
           min(date_added) as primera, max(date_added) as ultima
    from bronze
""").df()

→ **Silver deduplica por `id`** y descarta filas sin id o sin fecha, por si una reingesta repite URLs. Las que no tienen etiquetas quedan como `sin_familia` / `desconocido`.

### 2.2 ¿Sirve `threat` para distinguir amenazas?

In [ ]:
con.sql("""
    select threat, count(*) as urls, round(100.0 * count(*) / sum(count(*)) over (), 2) as pct
    from bronze group by 1 order by 2 desc
""").df()

→ Un solo valor, `malware_download`: **no distingue nada**. Por eso silver deduce dos dimensiones nuevas de las etiquetas: la **familia de malware** y el **tipo de archivo**.

### 2.3 ¿Qué hay en las etiquetas?

In [ ]:
etiquetas = con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select tag, count(*) as urls from t where tag <> '' group by 1 order by 2 desc limit 40
""").df()
etiquetas

Las etiquetas mezclan la familia (`Mozi`, `mirai`, `LuaJIT-loader`) con el tipo de archivo (`elf`, `exe`, `zip`), la arquitectura (`32-bit`, `mips`, `arm`), el lugar donde estaba alojado (`github`, `discord`), el país y etiquetas de quien reporta (`dropped-by-*`, `ua-*`). Y aparecen cosas peores:

In [ ]:
con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select case
             when regexp_matches(tag, '^[0-9]{1,3}([.-][0-9]{1,3}){3}([.:-][0-9]{1,5})?$') then 'IP'
             when regexp_matches(lower(tag), '[.-](com|net|org|xyz|top|site|space|to|ru|cn)$') then 'dominio'
             when regexp_matches(lower(tag), '^[0-9a-f]{6,}$') then 'pedazo de hash'
             when lower(tag) in ('cowrie', 'honeypot', 'ssh', 'telnet') then 'honeypot / protocolo'
           end as tipo_de_ruido,
           count(distinct tag) as etiquetas_distintas, count(*) as urls,
           any_value(tag) as ejemplo
    from t
    group by 1 having tipo_de_ruido is not null order by 3 desc
""").df()

→ **Silver toma como familia la primera etiqueta que no es ninguna de esas cosas.** Las IPs, los dominios, los honeypots y los hashes los encontramos recién con datos reales, cuando aparecieron como "familias" en el tablero. El test `assert_familia_no_es_tag_generico` evita que vuelvan.

### 2.4 ¿La misma familia aparece escrita distinto?

In [ ]:
con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select lower(tag) as familia, list(distinct tag) as grafias, count(*) as urls
    from t where tag <> ''
    group by 1 having count(distinct tag) > 1
    order by 3 desc limit 10
""").df()

→ `Mozi` / `mozi`, `Mirai` / `mirai`, ... **Silver unifica con la grafía más frecuente**, si no una misma familia se partiría en dos en el ranking.

### 2.5 ¿Cuántas URLs hay por mes? (cobertura)

In [ ]:
por_mes = con.sql("""
    select strftime(date_added, '%Y-%m') as mes, count(*) as urls
    from bronze group by 1 order by 1
""").df()
print(por_mes.tail(14).to_string(index=False))
print(f"\nMeses con datos: {len(por_mes)} · meses con menos de 1.000 URLs: {(por_mes.urls < 1000).sum()}")

→ **Antes de agosto de 2026 los meses tienen muchas menos URLs:** el export de URLhaus conserva bien solo los últimos uno o dos meses y ya no publica las URLs viejas. Es un hueco del origen, no del pipeline, y está documentado en la governance. El salto de octubre, en cambio, es real: un pico de campañas (ver el tablero).

### 2.6 ¿Quién reporta?

In [ ]:
con.sql("""
    select reporter, count(*) as urls, round(100.0 * count(*) / sum(count(*)) over (), 1) as pct,
           min(date_added)::date as desde, max(date_added)::date as hasta
    from bronze group by 1 order by 2 desc limit 8
""").df()

→ Unos pocos reporteros cargan buena parte de las URLs, a veces miles en pocos días. **Un pico en el tablero puede ser un reportero muy activo y no una campaña nueva**: queda como limitación en la governance. Además, `reporter` identifica a una persona o equipo: se queda en silver y no se publica.

## 3. Historial de versiones (*time travel*)

Delta guarda en su log cada operación sobre la tabla, y en principio permite leer cualquier versión anterior.

In [ ]:
historia = pd.DataFrame(dt.history(limit=10))
historia["timestamp"] = pd.to_datetime(historia["timestamp"], unit="ms", utc=True)
historia[[c for c in ["version", "timestamp", "operation", "operationParameters"] if c in historia.columns]]

**Las versiones viejas de bronze ya no se pueden leer.** Después de escribir, `extraer_bronze` hace `vacuum` con retención 0, porque dbt lee los `.parquet` directamente y sin `vacuum` leería también archivos que ya no son parte de la tabla. Eso borra los archivos de las versiones anteriores: el log las recuerda, pero sus datos ya no están. Lo sufrimos el 18/11/2026, cuando un reproceso borró URLs y el *time travel* no tenía a dónde volver (ver `docs/INCIDENTE_2026-11-18_backfill.md`).

In [ ]:
anterior = max(dt.version() - 1, 0)
try:
    n = DeltaTable(BRONZE, version=anterior, storage_options=STORAGE_OPTIONS).to_pyarrow_table(columns=["id"]).num_rows
    print(f"Versión {anterior}: {n} filas")
except Exception as e:
    print(f"La versión {anterior} no se puede leer: {type(e).__name__}. Sus archivos los borró el vacuum.")

Para ver el *time travel* funcionando, una tabla de demostración aparte (no toca bronze): se escribe dos veces y se lee la versión 0.

In [ ]:
from deltalake import write_deltalake
import pyarrow as pa

DEMO = f"s3://{BUCKET}/demo/time_travel"
write_deltalake(DEMO, pa.table({"id": [1, 2, 3], "estado": ["online"] * 3}),
                mode="overwrite", storage_options=STORAGE_OPTIONS)
write_deltalake(DEMO, pa.table({"id": [4, 5], "estado": ["offline"] * 2}),
                mode="append", storage_options=STORAGE_OPTIONS)

actual = DeltaTable(DEMO, storage_options=STORAGE_OPTIONS)
inicio = actual.version() - 1
v0 = DeltaTable(DEMO, version=inicio, storage_options=STORAGE_OPTIONS)
print(f"Versión actual ({actual.version()}): {actual.to_pyarrow_table().num_rows} filas")
print(f"Versión {inicio} (antes del append): {v0.to_pyarrow_table().num_rows} filas")

## Cierre

DuckDB sobre Delta permite explorar el lake con SQL sin moverlo a una base. Lo que encontramos acá (un solo `threat`, etiquetas con ruido, grafías distintas, huecos de cobertura, reporteros dominantes) es lo que limpia silver y lo que documenta la governance. Pendiente de mejora: si dbt leyera la tabla Delta con `delta_scan` en vez de los `.parquet` sueltos, se podría guardar historia de versiones (sacar el `vacuum` con retención 0) y tener *time travel* real sobre bronze.